<a href="https://colab.research.google.com/github/Jhairzp27/VitalsFlow/blob/main/03_Modelado_y_Carga_SQL_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implementación


## Carga de archivos

En esta etapa, se realiza la lectura del conjunto de datos previamente procesado.
Se utiliza la librería pandas para importar el archivo CSV y se verifica la
integridad de la carga mediante la impresión de las dimensiones del dataframe.

In [22]:
import pandas as pd
import sqlite3

df_limpio = pd.read_csv('healthcare_cleaned.csv')
print(f"Datos cargados en memoria: {df_limpio.shape}")


Datos cargados en memoria: (50000, 16)


## Establecimiento de conexión
1. Se procede con la creación de una conexión hacia el motor de base de datos SQLite.
2. Se define el nombre del archivo de base de datos y se inicializa el cursor necesario para la ejecución de comandos SQL posteriores.

In [23]:
db_name = 'healthcare_warehouse.db'
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

## Migración de datos a SQL
Se ejecuta la transferencia de la información desde la memoria hacia el almacén de datos. Se emplea un modelo de tabla plana para maximizar el rendimiento en procesos analíticos, sobrescribiendo la tabla si ya existiera en el sistema.

In [24]:
table_name = 'hospital_records'
df_limpio.to_sql(table_name, conn, if_exists='replace', index=False)
print(f"¡Éxito! Datos insertados en la tabla '{table_name}' de la base de datos '{db_name}'.")


¡Éxito! Datos insertados en la tabla 'hospital_records' de la base de datos 'healthcare_warehouse.db'.


## Análisis y validación mediante consultas
Se lleva a cabo una validación de los datos cargados mediante la ejecución de una consulta SQL. Esta operación agrupa los registros por condición médica, calcula  el total de pacientes y el promedio de facturación por categoría.

In [25]:
print("\n--- PRUEBA DE CONSULTA SQL ---")
query = f"""
    SELECT
        \"Medical Condition\",
        COUNT(*) as Total_Patients,
        ROUND(AVG(\"Billing Amount\"), 2) as Avg_Billing
    FROM {table_name}
    GROUP BY \"Medical Condition\"
    ORDER BY Total_Patients DESC;
"""
resultado_sql = pd.read_sql_query(query, conn)
display(resultado_sql)



--- PRUEBA DE CONSULTA SQL ---


,Medical Condition,Total_Patients,Avg_Billing
0,Arthritis,8439,25487.42
1,Diabetes,8384,25656.05
2,Hypertension,8319,25496.03
3,Cancer,8294,25237.07
4,Obesity,8292,25786.13
5,Asthma,8272,25683.61


## Cierre de sesión
Como etapa final del procedimiento, se cierra formalmente la conexión con la base de datos para liberar los recursos del sistema y asegurar la persistencia.

In [26]:
conn.close()

# Prueba ácida

### Prueba de Volumen (Reconciliación)

Se Verifica que no se perdip ni se duplicó registros en la transferencia a SQL.

In [27]:
import sqlite3
# Re-estableciendo la conexión cerrada anteriormente
conn = sqlite3.connect('healthcare_warehouse.db')

print("--- INICIANDO BATERÍA DE PRUEBAS DE CALIDAD (QA) ---\n")
query_volumen = 'SELECT COUNT(*) FROM hospital_records;'
total_filas = pd.read_sql_query(query_volumen, conn).iloc[0,0]
print(f"Prueba 1 (Volumen): {'✅ PASÓ' if total_filas == 50000 else '❌ FALLÓ'} -> Hay {total_filas} registros en la BD.")

--- INICIANDO BATERÍA DE PRUEBAS DE CALIDAD (QA) ---

Prueba 1 (Volumen): ✅ PASÓ -> Hay 50000 registros en la BD.


## Prueba Lógica de Negocio

*Enfocado en fechas.*

Es imposible que alguien sea dado de alta (Discharge) ANTES de ser admitido (Admission).

In [28]:
query_fechas = 'SELECT COUNT(*) FROM hospital_records WHERE "Days Hospitalized" < 0;'
errores_fechas = pd.read_sql_query(query_fechas, conn).iloc[0,0]
print(f"Prueba 2 (Lógica de Fechas): {'✅ PASÓ' if errores_fechas == 0 else '❌ FALLÓ'} -> {errores_fechas} pacientes tienen alta médica antes del ingreso.")

Prueba 2 (Lógica de Fechas): ✅ PASÓ -> 0 pacientes tienen alta médica antes del ingreso.


## Prueba de Integridad Financiera

Ya no deberían existir montos en negativo.

In [29]:
query_finanzas = 'SELECT MIN("Billing Amount") FROM hospital_records;'
min_factura = pd.read_sql_query(query_finanzas, conn).iloc[0,0]
print(f"Prueba 3 (Finanzas): {'✅ PASÓ' if min_factura > 0 else '❌ FALLÓ'} -> La factura más baja es ${min_factura}")

Prueba 3 (Finanzas): ✅ PASÓ -> La factura más baja es $9.24


## Prueba de Nulos Post-Carga
A veces al pasar de Pandas a SQL se generan nulos (NaN a NULL). Verificamos que no pasó.

In [30]:
query_nulos = 'SELECT COUNT(*) FROM hospital_records WHERE "Name" IS NULL OR "Billing Amount" IS NULL;'
nulos_sql = pd.read_sql_query(query_nulos, conn).iloc[0,0]
print(f"Prueba 4 (Nulos SQL): {'✅ PASÓ' if nulos_sql == 0 else '❌ FALLÓ'} -> {nulos_sql} nulos detectados en columnas clave.")

print("----------------------------------------------------")

Prueba 4 (Nulos SQL): ✅ PASÓ -> 0 nulos detectados en columnas clave.
----------------------------------------------------


# Casos especiales

In [31]:
print("--- ANÁLISIS DE CASOS ESPECIALES ---\n")

# Pregunta 1: Verificación de pacientes sin alta médica
query_sin_alta = 'SELECT COUNT(*) FROM hospital_records WHERE "Discharge Date" IS NULL;'
sin_alta = pd.read_sql_query(query_sin_alta, conn).iloc[0,0]
print(f"Pacientes internados actualmente (sin fecha de alta): {sin_alta}")
print("(Esto confirma que el dataset sintético cerró el ciclo de todos los pacientes).\n Es decir, todos fueron dados de alta\n")

# Pregunta 2: Pacientes que han venido más de una vez (Pacientes Recurrentes)
query_recurrentes = """
    SELECT
        "Name" as Nombre_Paciente,
        COUNT(*) as Total_Visitas
    FROM hospital_records
    GROUP BY "Name"
    HAVING COUNT(*) > 1
    ORDER BY Total_Visitas DESC
    LIMIT 10;
"""
recurrentes = pd.read_sql_query(query_recurrentes, conn)

print("---Pacientes recurrentes:---\n")
if len(recurrentes) > 0:
    print(f"¡Sí! Encontramos pacientes que han venido varias veces. Aquí está el Top 10:")
    display(recurrentes)
else:
    print("No hay pacientes recurrentes. Todos vinieron una sola vez.")

# Finalizar conexión
conn.close()

--- ANÁLISIS DE CASOS ESPECIALES ---

Pacientes internados actualmente (sin fecha de alta): 0
(Esto confirma que el dataset sintético cerró el ciclo de todos los pacientes).
 Es decir, todos fueron dados de alta

---Pacientes recurrentes:---

¡Sí! Encontramos pacientes que han venido varias veces. Aquí está el Top 10:


,Nombre_Paciente,Total_Visitas
0,Michael Williams,22
1,Robert Smith,21
2,Michael Smith,21
3,James Smith,17
4,John Smith,16
5,Matthew Smith,15
6,James Brown,15
7,Michael Jones,14
8,James Williams,14
9,Christopher Smith,14
